# Phase 5 - Merge JSONL datasets

Notebook này xử lý 2 nhiệm vụ:

1. Nối toàn bộ file `.jsonl` trong `dataset/Chunk_id` thành một file JSONL thống nhất.
2. Nối toàn bộ file `.jsonl` dạng `messages` trong `dataset/Samples`, không lấy file `structured`, thành một file JSONL thống nhất.

Output sẽ nằm riêng trong:

```text
dataset/merged_jsonl/
```

## 0. Imports và cấu hình đường dẫn

In [1]:
from pathlib import Path
import json
import re


def existing_child(parent, *names):
    """Lấy thư mục/file con theo tên, chịu được khác biệt hoa/thường giữa Windows và Linux/Colab."""
    parent = Path(parent)
    for name in names:
        candidate = parent / name
        if candidate.exists():
            return candidate

    if parent.exists():
        wanted = {name.lower() for name in names}
        for child in parent.iterdir():
            if child.name.lower() in wanted:
                return child

    return parent / names[0]


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        dataset_dir = existing_child(candidate, "dataset", "Dataset")
        training_dir = existing_child(candidate, "Training", "training")
        if dataset_dir.exists() and training_dir.exists():
            return candidate

    raise FileNotFoundError("Không tìm thấy project root chứa cả dataset/ và Training/.")


PROJECT_ROOT = find_project_root()
DATASET_DIR = existing_child(PROJECT_ROOT, "dataset", "Dataset")
CHUNK_ID_DIR = existing_child(DATASET_DIR, "Chunk_id", "chunk_id")
SAMPLES_DIR = existing_child(DATASET_DIR, "Samples", "samples")

OUTPUT_DIR = DATASET_DIR / "merged_jsonl"
CHUNK_OUTPUT_PATH = OUTPUT_DIR / "all_chunk_id.jsonl"
MESSAGES_OUTPUT_PATH = OUTPUT_DIR / "all_messages.jsonl"

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: C:\Users\PC\Chatbot_answering_vietnamese_history
Dataset dir: C:\Users\PC\Chatbot_answering_vietnamese_history\dataset
Output dir: C:\Users\PC\Chatbot_answering_vietnamese_history\dataset\merged_jsonl


## 1. Hàm nối JSONL dùng chung

In [2]:
def natural_key(path):
    text = str(path).replace("\\", "/")
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", text)]


def concat_jsonl(input_files, output_file, validate_json=True):
    input_files = list(input_files)
    output_file = Path(output_file)

    if not input_files:
        raise ValueError("Không tìm thấy file JSONL đầu vào để nối.")

    output_file.parent.mkdir(parents=True, exist_ok=True)

    total_lines = 0
    summary = []

    with output_file.open("w", encoding="utf-8", newline="\n") as out_f:
        for source_path in input_files:
            source_path = Path(source_path)
            file_lines = 0

            with source_path.open("r", encoding="utf-8") as in_f:
                for line_number, raw_line in enumerate(in_f, start=1):
                    line = raw_line.rstrip("\r\n")
                    if not line.strip():
                        continue

                    if validate_json:
                        try:
                            json.loads(line)
                        except json.JSONDecodeError as exc:
                            rel_path = source_path.relative_to(PROJECT_ROOT)
                            raise ValueError(f"JSON lỗi ở {rel_path}, dòng {line_number}: {exc}") from exc

                    out_f.write(line + "\n")
                    file_lines += 1

            summary.append({
                "file": str(source_path.relative_to(PROJECT_ROOT)),
                "lines": file_lines,
            })
            total_lines += file_lines

    print(f"Đã nối {len(input_files)} file -> {output_file.relative_to(PROJECT_ROOT)}")
    print(f"Tổng số dòng JSONL: {total_lines:,}")
    return summary

## 2. Nhiệm vụ 1 - Nối các file trong `dataset/Chunk_id`

In [3]:
chunk_id_files = sorted(CHUNK_ID_DIR.glob("*.jsonl"), key=natural_key)

print(f"Tìm thấy {len(chunk_id_files)} file JSONL trong {CHUNK_ID_DIR.relative_to(PROJECT_ROOT)}")
for path in chunk_id_files:
    print("-", path.relative_to(PROJECT_ROOT))

Tìm thấy 29 file JSONL trong dataset\Chunk_id
- dataset\Chunk_id\pack_bach_dang_ngo_quyen.jsonl
- dataset\Chunk_id\pack_cach_mang_thang_tam.jsonl
- dataset\Chunk_id\pack_can_vuong_phap_thuoc.jsonl
- dataset\Chunk_id\pack_chien_dich_viet_bac_bien_gioi_1950.jsonl
- dataset\Chunk_id\pack_chong_my_1975.jsonl
- dataset\Chunk_id\pack_dien_bien_phu.jsonl
- dataset\Chunk_id\pack_dong_du_duy_tan_phan_boi_chau.jsonl
- dataset\Chunk_id\pack_duong_truong_son_mau_than_1968.jsonl
- dataset\Chunk_id\pack_hai_ba_trung_ba_trieu.jsonl
- dataset\Chunk_id\pack_hiep_dinh_geneve_1954.jsonl
- dataset\Chunk_id\pack_khoi_nghia_yen_the_hoang_hoa_tham.jsonl
- dataset\Chunk_id\pack_khuc_duong_thoi_tu_chu.jsonl
- dataset\Chunk_id\pack_lam_son_le_loi.jsonl
- dataset\Chunk_id\pack_le_so_hong_duc_le_thanh_tong.jsonl
- dataset\Chunk_id\pack_loan_12_su_quan_dinh_bo_linh.jsonl
- dataset\Chunk_id\pack_mac_le_trung_hung_trinh_nguyen.jsonl
- dataset\Chunk_id\pack_mai_hac_de_phung_hung.jsonl
- dataset\Chunk_id\pack_minh_thu

In [4]:
chunk_summary = concat_jsonl(
    input_files=chunk_id_files,
    output_file=CHUNK_OUTPUT_PATH,
    validate_json=True,
)

chunk_summary[:5]

Đã nối 29 file -> dataset\merged_jsonl\all_chunk_id.jsonl
Tổng số dòng JSONL: 500


[{'file': 'dataset\\Chunk_id\\pack_bach_dang_ngo_quyen.jsonl', 'lines': 12},
 {'file': 'dataset\\Chunk_id\\pack_cach_mang_thang_tam.jsonl', 'lines': 12},
 {'file': 'dataset\\Chunk_id\\pack_can_vuong_phap_thuoc.jsonl', 'lines': 12},
 {'file': 'dataset\\Chunk_id\\pack_chien_dich_viet_bac_bien_gioi_1950.jsonl',
  'lines': 20},
 {'file': 'dataset\\Chunk_id\\pack_chong_my_1975.jsonl', 'lines': 12}]

## 3. Nhiệm vụ 2 - Nối các file `messages`, không lấy `structured`

In [5]:
message_files = sorted(
    [
        path
        for path in SAMPLES_DIR.rglob("*.jsonl")
        if "messages" in path.stem.lower()
        and "structured" not in path.stem.lower()
    ],
    key=natural_key,
)

print(f"Tìm thấy {len(message_files)} file JSONL dạng messages trong {SAMPLES_DIR.relative_to(PROJECT_ROOT)}")
for path in message_files:
    print("-", path.relative_to(PROJECT_ROOT))

Tìm thấy 29 file JSONL dạng messages trong dataset\Samples
- dataset\Samples\Pack1\bach_dang_ngo_quyen_rag_sft_messages_20.jsonl
- dataset\Samples\Pack2\cach_mang_thang_tam_rag_sft_20_messages.jsonl
- dataset\Samples\Pack3\can_vuong_phap_thuoc_rag_sft_20_messages.jsonl
- dataset\Samples\Pack4\chong_my_1975_rag_sft_20_messages.jsonl
- dataset\Samples\Pack5\dien_bien_phu_rag_sft_20_messages.jsonl
- dataset\Samples\Pack6\lam_son_le_loi_rag_sft_20_messages.jsonl
- dataset\Samples\Pack7\nha_dinh_tien_le_rag_sft_20_messages.jsonl
- dataset\Samples\Pack8\nha_ly_rag_sft_20_messages.jsonl
- dataset\Samples\Pack9\nha_tran_mong_nguyen_rag_sft_20_messages.jsonl
- dataset\Samples\Pack10\tay_son_quang_trung_rag_sft_20_messages.jsonl
- dataset\Samples\Pack11\chien_dich_viet_bac_bien_gioi_1950_rag_sft_40_messages.jsonl
- dataset\Samples\Pack12\dong_du_duy_tan_phan_boi_chau_rag_sft_40_messages.jsonl
- dataset\Samples\Pack13\duong_truong_son_mau_than_1968_rag_sft_40_messages.jsonl
- dataset\Samples\Pack

In [6]:
messages_summary = concat_jsonl(
    input_files=message_files,
    output_file=MESSAGES_OUTPUT_PATH,
    validate_json=True,
)

messages_summary[:5]

Đã nối 29 file -> dataset\merged_jsonl\all_messages.jsonl
Tổng số dòng JSONL: 960


[{'file': 'dataset\\Samples\\Pack1\\bach_dang_ngo_quyen_rag_sft_messages_20.jsonl',
  'lines': 20},
 {'file': 'dataset\\Samples\\Pack2\\cach_mang_thang_tam_rag_sft_20_messages.jsonl',
  'lines': 20},
 {'file': 'dataset\\Samples\\Pack3\\can_vuong_phap_thuoc_rag_sft_20_messages.jsonl',
  'lines': 20},
 {'file': 'dataset\\Samples\\Pack4\\chong_my_1975_rag_sft_20_messages.jsonl',
  'lines': 20},
 {'file': 'dataset\\Samples\\Pack5\\dien_bien_phu_rag_sft_20_messages.jsonl',
  'lines': 20}]

## 4. Kết quả mong đợi

Sau khi chạy notebook, bạn sẽ có 2 file mới:

```text
dataset/merged_jsonl/all_chunk_id.jsonl
dataset/merged_jsonl/all_messages.jsonl
```